# Kansas Economic and Workforce Trends Analysis

## Project Goal

This project analyzes economic and workforce conditions across all 105 Kansas counties using data from the U.S. Census Bureau American Community Survey (ACS) and the Bureau of Labor Statistics (BLS) Local Area Unemployment Statistics (LAUS).

The analysis focuses on county-level differences in:

- Population
- Median household income
- Labor force participation
- Unemployment
- Poverty
- Educational attainment

The project also examines relationships among these indicators using correlation analysis, SQL queries, and multiple linear regression.

## Main Research Question

How do economic and workforce conditions vary across Kansas counties, and which factors are associated with higher or lower unemployment rates?


## Data Sources

### U.S. Census Bureau — American Community Survey (ACS) 5-Year Estimates, 2024

The Census ACS data provides county-level demographic and socioeconomic indicators used in this project, including:

- Total population
- Median household income
- Civilian labor force
- Population age 16 and older
- Unemployed population
- Poverty status
- Population age 25 and older
- Bachelor's, master's, professional, and doctorate degree attainment

These variables were retrieved through the Census API for all 105 Kansas counties.

### Bureau of Labor Statistics — Local Area Unemployment Statistics (LAUS), 2024 Annual Averages

The BLS LAUS county dataset provides:

- Labor force
- Employment
- Unemployment
- Unemployment rate

The national county file was filtered to Kansas using State FIPS Code 20.

### Geographic Matching

Census and BLS data were merged using:

- State FIPS code
- County FIPS code

This produced a final dataset containing all 105 Kansas counties.

## Data Preparation

The Census ACS and BLS LAUS datasets required several preparation steps before analysis.

### Census Data Preparation

The Census API data was processed to:

- Rename Census variable codes into understandable column names.
- Convert numeric variables from text to numeric data types.
- Calculate labor force participation rate.
- Calculate unemployment rate.
- Calculate poverty rate.
- Combine bachelor's, master's, professional, and doctorate degree counts.
- Calculate the percentage of adults age 25 and older with a bachelor's degree or higher.

### BLS Data Preparation

The BLS LAUS county file was processed to:

- Correct the Excel header structure.
- Filter the national dataset to Kansas counties.
- Select labor force, employment, unemployment, and unemployment rate measures.
- Standardize State and County FIPS codes.

### Data Integration and Validation

The Census and BLS datasets were merged using State and County FIPS codes.

The final dataset contained:

- **105 Kansas counties**
- **26 variables**
- **0 duplicate counties**
- **0 missing values in the analyzed variables**

FIPS codes were used for the merge instead of county names because the Census and BLS datasets use different county-name formats.

import pandas as pd

# Load the final merged Kansas dataset
file_path = "../data/processed/kansas_economic_workforce_2024.csv"
df = pd.read_csv(file_path)

# Display the first five rows
df.head()







## Data Preparation

The Census ACS and BLS LAUS datasets required several preparation steps before analysis.

### Census Data Preparation

The Census API data was processed to:

- Rename Census variable codes into understandable column names.
- Convert numeric variables from text to numeric data types.
- Calculate labor force participation rate.
- Calculate unemployment rate.
- Calculate poverty rate.
- Combine bachelor's, master's, professional, and doctorate degree counts.
- Calculate the percentage of adults age 25 and older with a bachelor's degree or higher.

### BLS Data Preparation

The BLS LAUS county file was processed to:

- Correct the Excel header structure.
- Filter the national dataset to Kansas counties.
- Select labor force, employment, unemployment, and unemployment rate measures.
- Standardize State and County FIPS codes.

### Data Integration and Validation

The Census and BLS datasets were merged using State and County FIPS codes.

The final dataset contained:

- **105 Kansas counties**
- **26 variables**
- **0 duplicate counties**
- **0 missing values in the analyzed variables**

FIPS codes were used for the merge instead of county names because the Census and BLS datasets use different county-name formats.



In [ ]:

import pandas as pd

# Load the final merged Kansas dataset
file_path = "../data/processed/kansas_economic_workforce_2024.csv"
df = pd.read_csv(file_path)

# Display the first five rows
df.head()



In [ ]:
# Check the size of the final dataset
print("Dataset shape:", df.shape)

# Check for duplicate counties
print("Duplicate counties:", df["county_fips"].duplicated().sum())

# Check missing values in the main analysis variables
analysis_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

print("\nMissing values:")
print(df[analysis_columns].isnull().sum())

## Exploratory Data Analysis

The next step examines differences in unemployment across Kansas counties. Ranking counties by unemployment rate helps identify areas experiencing relatively higher and lower unemployment in the 2024 BLS LAUS data.


In [ ]:
# Select county name and unemployment rate
unemployment_analysis = df[
    ["county_name_census", "unemployment_rate_bls"]
].copy()

# Rename columns for easier reading
unemployment_analysis.columns = [
    "County",
    "Unemployment Rate"
]

# Display the 10 counties with the highest unemployment rates
top_10_unemployment = unemployment_analysis.sort_values(
    by="Unemployment Rate",
    ascending=False
).head(10)

top_10_unemployment

### Unemployment Findings

The county-level analysis shows meaningful variation in unemployment across Kansas.

- **Bourbon County** had the highest unemployment rate in the dataset at **5.3%**.
- **Woodson County** and **Linn County** followed at **4.9%**.
- Other counties with relatively high unemployment included Cowley, Geary, Wyandotte, Harvey, Elk, Wilson, and Lincoln counties.
- The average unemployment rate across the 105 Kansas counties was approximately **3.4%**.

These results show that statewide or average measures can hide important differences between individual counties. Examining county-level data makes it possible to identify areas experiencing relatively greater labor-market challenges.

## Correlation Analysis

Correlation analysis is used to examine the direction and strength of relationships among the main economic and workforce indicators.

The analysis includes:

- Unemployment rate
- Poverty rate
- Median household income
- Bachelor's-or-higher educational attainment rate
- Labor-force participation rate

Correlation values range from **-1 to +1**. A positive value indicates that two variables tend to move in the same direction, while a negative value indicates that they tend to move in opposite directions. Correlation describes association and does not establish causation.

In [ ]:
# Select variables for correlation analysis
correlation_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

# Calculate the correlation matrix
correlation_matrix = df[correlation_columns].corr().round(2)

# Display the correlation matrix
correlation_matrix

In [ ]:
correlation_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

correlation_matrix = df[correlation_columns].corr().round(2)

correlation_matrix


## Multiple Linear Regression

A multiple linear regression model was used to examine how several county-level characteristics are associated with unemployment when considered together.

### Dependent Variable

- BLS unemployment rate

### Independent Variables

- Poverty rate
- Median household income
- Bachelor's-or-higher educational attainment rate
- Labor-force participation rate

The regression estimates the association between each predictor and unemployment while holding the other variables in the model constant.


In [ ]:
import statsmodels.api as sm

# Define the dependent variable
y = df["unemployment_rate_bls"]

# Define the independent variables
X = df[
    [
        "poverty_rate",
        "median_household_income",
        "bachelors_or_higher_rate",
        "labor_force_participation_rate"
    ]
]

# Add the regression intercept
X = sm.add_constant(X)

# Fit the multiple linear regression model
model = sm.OLS(y, X).fit()

# Display the regression results
model.summary()


### Regression Findings

The multiple linear regression model was statistically significant overall (**F-test p = 0.000660**) and explained approximately **17.5% of the variation in county unemployment rates (R² = 0.175)**.

Among the four predictors:

- **Labor-force participation rate** was statistically significant (**coefficient = -0.0378, p = 0.002**). Holding the other variables constant, a 1 percentage-point increase in labor-force participation was associated with approximately a 0.038 percentage-point decrease in the unemployment rate.
- **Poverty rate** had a positive coefficient (**0.0275**) but was not statistically significant at the 5% level (**p = 0.085**).
- **Median household income** was not statistically significant (**p = 0.590**).
- **Bachelor's-or-higher educational attainment** was not statistically significant (**p = 0.848**).

The results suggest that labor-force participation had the clearest independent association with county unemployment among the variables included in this model. However, the relatively low R² indicates that much of the variation in county unemployment is associated with factors not included in this model.

These results describe statistical associations and should not be interpreted as evidence of causal relationships.


## Multicollinearity Diagnostic

Variance Inflation Factor (VIF) was used to check whether the independent variables in the regression model were highly correlated with one another.

High multicollinearity can make regression coefficients unstable and difficult to interpret. As a general diagnostic guideline, VIF values below 5 indicate that serious multicollinearity is not evident.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each regression variable
vif_data = pd.DataFrame()

vif_data["Variable"] = X.columns

vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif_data

### VIF Findings

The VIF results indicate that multicollinearity is not a major concern in the regression model.

- Poverty rate: **VIF = 1.38**
- Median household income: **VIF = 1.78**
- Bachelor's-or-higher rate: **VIF = 1.32**
- Labor-force participation rate: **VIF = 1.33**

All predictor VIF values are well below 5. This suggests that the independent variables are not excessively correlated with one another and can reasonably be included together in the regression model.

The regression output also reported a large condition number. Because median household income is measured in dollars while the other predictors are percentages, differences in variable scale can contribute to a large condition number. The VIF results provide additional evidence that severe predictor multicollinearity is not apparent.


## Visual Analysis

The following visualizations provide additional context for the relationships identified in the descriptive, correlation, and regression analyses.

The charts examine:

- Poverty and median household income
- Unemployment and labor-force participation
- Educational attainment and household income
- Counties with the highest unemployment rates


In [ ]:
from IPython.display import Image, display

# Display poverty vs. median household income chart
display(
    Image(
        filename="../outputs/charts/poverty_vs_income.png"
    )
)

### Poverty and Household Income

The visualization shows a general negative relationship between poverty and median household income across Kansas counties.

This pattern is consistent with the correlation analysis, which found a correlation of **-0.49** between poverty rate and median household income. Counties with higher median household incomes generally tended to have lower poverty rates.

The relationship is not perfect, however, because counties with similar income levels can still have different poverty rates. This suggests that household income alone does not fully explain county-level differences in poverty.

The observed relationship represents an association and does not establish causation.

In [ ]:
# Display unemployment vs. labor-force participation chart
display(
    Image(
        filename="../outputs/charts/unemployment_vs_labor_force.png"
    )
)

In [ ]:
# Display educational attainment vs. household income chart
display(
    Image(
        filename="../outputs/charts/education_vs_income.png"
    )
)

### Educational Attainment and Household Income

The visualization shows a positive relationship between bachelor's-or-higher educational attainment and median household income across Kansas counties.

The correlation analysis found a correlation of **+0.40**, indicating that counties with higher educational attainment tended to have higher median household incomes.

For example, Johnson County had the highest bachelor's-or-higher rate among the counties identified in the ranking analysis at **57.56%**, along with a median household income of **$109,208**.

However, educational attainment was not statistically significant as an independent predictor of unemployment in the multiple regression model (**p = 0.848**). This demonstrates an important distinction: education may be associated with household income while not showing a statistically significant independent association with unemployment in this particular regression specification.

These findings represent county-level associations and do not establish causal relationships.

In [ ]:
# Display the Top 10 Kansas counties by unemployment rate
display(
    Image(
        filename="../outputs/charts/top_10_unemployment.png"
    )
)

### Counties with the Highest Unemployment Rates

The county ranking shows that unemployment conditions varied considerably across Kansas in the 2024 BLS LAUS data.

**Bourbon County** had the highest unemployment rate at **5.3%**, followed by **Woodson County** and **Linn County**, both at **4.9%**.

The average unemployment rate across all 105 counties was approximately **3.4%**, meaning the counties at the top of the ranking were notably above the county average.

Several counties with relatively high unemployment also appeared in the earlier SQL analysis of counties with both above-average unemployment and above-average poverty. This demonstrates the value of examining multiple economic indicators rather than relying on unemployment alone.

These county rankings are descriptive and identify areas that may warrant further investigation; they do not by themselves explain why unemployment differs across counties.

## SQL Analysis

The cleaned county-level dataset was loaded into a SQLite database to demonstrate SQL-based analysis.

The SQL analysis focused on four business and policy-oriented questions:

1. Which Kansas counties had the highest unemployment rates?
2. Which counties had both above-average unemployment and above-average poverty?
3. Which counties had the lowest labor-force participation rates?
4. Which counties had the highest bachelor's-or-higher educational attainment rates, and what were their median household incomes?

SQL complements the Python analysis by demonstrating how structured queries can be used to filter, rank, and compare county-level economic indicators.

In [ ]:
import sqlite3

# Connect to the project SQLite database
connection = sqlite3.connect(
    "../data/processed/kansas_economic_workforce.db"
)

print("Database connected successfully.")

In [ ]:
# Show all tables available in the SQLite database

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

tables

In [ ]:
# SQL query: Counties with above-average unemployment
# and above-average poverty

query = """
SELECT
    county_name,
    unemployment_rate,
    poverty_rate
FROM kansas_county_economic_data
WHERE unemployment_rate > (
    SELECT AVG(unemployment_rate)
    FROM kansas_county_economic_data
)
AND poverty_rate > (
    SELECT AVG(poverty_rate)
    FROM kansas_county_economic_data
)
ORDER BY unemployment_rate DESC, poverty_rate DESC;
"""

sql_high_unemployment_poverty = pd.read_sql_query(
    query,
    connection
)

sql_high_unemployment_poverty

### SQL Findings: Unemployment and Poverty

The SQL analysis identified Kansas counties with both above-average unemployment and above-average poverty.

Examples included:

- Bourbon County
- Woodson County
- Geary County
- Cowley County
- Wyandotte County
- Wilson County
- Chautauqua County
- Brown County

This combined screening is more informative than ranking unemployment alone because it identifies counties experiencing challenges across multiple economic indicators.

The SQL approach also demonstrates how subqueries can be used to compare each county with statewide county-level averages and isolate areas that may warrant additional investigation.

In [ ]:
# SQL query: Counties with the lowest labor-force participation rates

query = """
SELECT
    county_name,
    labor_force_participation_rate,
    unemployment_rate,
    poverty_rate
FROM kansas_county_economic_data
ORDER BY labor_force_participation_rate ASC, county_name ASC
LIMIT 10;
"""

sql_lowest_participation = pd.read_sql_query(
    query,
    connection
)

sql_lowest_participation

### SQL Findings: Labor-Force Participation

The SQL results show that some Kansas counties had relatively low labor-force participation even when their unemployment rates were not especially high.

For example:

- **Norton County** had the lowest labor-force participation rate in the analysis at **45.97%**, while its unemployment rate was **2.9%**.
- **Ellsworth County** had labor-force participation of **50.76%** and unemployment of **3.8%**.
- **Woodson County** combined relatively low labor-force participation (**54.22%**) with relatively high unemployment (**4.9%**).

This illustrates why unemployment should not be interpreted by itself. A county can have a low unemployment rate while also having a relatively small share of its population participating in the labor force.

Examining both measures provides a more complete view of local workforce conditions.

In [ ]:
# SQL query: Counties with the highest educational attainment

query = """
SELECT
    county_name,
    bachelors_or_higher_rate,
    median_household_income
FROM kansas_county_economic_data
ORDER BY bachelors_or_higher_rate DESC,
         median_household_income DESC
LIMIT 10;
"""

sql_education_income = pd.read_sql_query(
    query,
    connection
)

sql_education_income

### SQL Findings: Educational Attainment and Income

The SQL ranking shows that counties with the highest educational attainment rates varied considerably in household income.

- **Johnson County** had the highest bachelor's-or-higher attainment rate at **57.56%** and a median household income of **$109,208**.
- **Douglas County** had an attainment rate of **53.19%** and median household income of **$69,746**.
- **Riley County** had an attainment rate of **48.91%** and median household income of **$61,098**.
- **Pottawatomie County** had an attainment rate of **38.19%** and median household income of **$92,325**.

The ranking is based primarily on educational attainment rather than household income. Therefore, these results should not be interpreted as a ranking of the highest-income Kansas counties.

Combined with the correlation analysis, the results show a positive county-level association between educational attainment and median household income, while also showing that counties with similar educational attainment levels can have different income levels.

## Key Findings

The analysis identified several important patterns in economic and workforce conditions across Kansas counties.

1. **Unemployment varied considerably across counties.**  
   The average county unemployment rate was approximately **3.4%**, while Bourbon County had the highest rate at **5.3%**. Woodson and Linn counties followed at **4.9%**.

2. **Labor-force participation was negatively associated with unemployment.**  
   The correlation between labor-force participation and unemployment was **-0.35**. In the multiple regression model, labor-force participation was the only predictor statistically significant at the 5% level (**coefficient = -0.0378, p = 0.002**).

3. **Low unemployment did not always indicate strong labor-force participation.**  
   Norton County had an unemployment rate of only **2.9%**, but its labor-force participation rate was **45.97%**, the lowest among the counties analyzed.

4. **Higher household income was associated with lower poverty.**  
   Poverty and median household income had a correlation of **-0.49**, one of the stronger relationships observed in the analysis.

5. **Educational attainment was positively associated with household income.**  
   Bachelor's-or-higher attainment and median household income had a correlation of **+0.40**. Johnson County had the highest educational attainment rate in the ranking at **57.56%** and a median household income of **$109,208**.

6. **The regression model explained only part of the differences in unemployment.**  
   The model had an **R² of 0.175**, indicating that the four included predictors explained approximately **17.5%** of the variation in county unemployment rates. Other economic, demographic, geographic, and industry-related factors may account for additional variation.

Overall, the findings demonstrate the importance of examining multiple indicators when evaluating local workforce conditions rather than relying on unemployment alone.

## Data Sources

### U.S. Census Bureau — American Community Survey (ACS) 5-Year Estimates, 2024

The Census ACS data provides county-level demographic and socioeconomic indicators used in this project, including:

- Total population
- Median household income
- Civilian labor force
- Population age 16 and older
- Unemployed population
- Poverty status
- Population age 25 and older
- Bachelor's, master's, professional, and doctorate degree attainment

These variables were retrieved through the Census API for all 105 Kansas counties.

### Bureau of Labor Statistics — Local Area Unemployment Statistics (LAUS), 2024 Annual Averages

The BLS LAUS county dataset provides:

- Labor force
- Employment
- Unemployment
- Unemployment rate

The national county file was filtered to Kansas using State FIPS Code 20.

### Geographic Matching

Census and BLS data were merged using:

- State FIPS code
- County FIPS code

This produced a final dataset containing all 105 Kansas counties.



## Data Preparation

The Census ACS and BLS LAUS datasets required several preparation steps before analysis.

### Census Data Preparation

The Census API data was processed to:

- Rename Census variable codes into understandable column names.
- Convert numeric variables from text to numeric data types.
- Calculate labor force participation rate.
- Calculate unemployment rate.
- Calculate poverty rate.
- Combine bachelor's, master's, professional, and doctorate degree counts.
- Calculate the percentage of adults age 25 and older with a bachelor's degree or higher.

### BLS Data Preparation

The BLS LAUS county file was processed to:

- Correct the Excel header structure.
- Filter the national dataset to Kansas counties.
- Select labor force, employment, unemployment, and unemployment rate measures.
- Standardize State and County FIPS codes.

### Data Integration and Validation

The Census and BLS datasets were merged using State and County FIPS codes.

The final dataset contained:

- **105 Kansas counties**
- **26 variables**
- **0 duplicate counties**
- **0 missing values in the analyzed variables**

FIPS codes were used for the merge instead of county names because the Census and BLS datasets use different county-name formats.



In [ ]:

import pandas as pd

# Load the final merged Kansas dataset
file_path = "../data/processed/kansas_economic_workforce_2024.csv"
df = pd.read_csv(file_path)

# Display the first five rows
df.head()



In [ ]:
# Check the size of the final dataset
print("Dataset shape:", df.shape)

# Check for duplicate counties
print("Duplicate counties:", df["county_fips"].duplicated().sum())

# Check missing values in the main analysis variables
analysis_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

print("\nMissing values:")
print(df[analysis_columns].isnull().sum())

## Exploratory Data Analysis

The next step examines differences in unemployment across Kansas counties. Ranking counties by unemployment rate helps identify areas experiencing relatively higher and lower unemployment in the 2024 BLS LAUS data.


In [ ]:
# Select county name and unemployment rate
unemployment_analysis = df[
    ["county_name_census", "unemployment_rate_bls"]
].copy()

# Rename columns for easier reading
unemployment_analysis.columns = [
    "County",
    "Unemployment Rate"
]

# Display the 10 counties with the highest unemployment rates
top_10_unemployment = unemployment_analysis.sort_values(
    by="Unemployment Rate",
    ascending=False
).head(10)

top_10_unemployment

### Unemployment Findings

The county-level analysis shows meaningful variation in unemployment across Kansas.

- **Bourbon County** had the highest unemployment rate in the dataset at **5.3%**.
- **Woodson County** and **Linn County** followed at **4.9%**.
- Other counties with relatively high unemployment included Cowley, Geary, Wyandotte, Harvey, Elk, Wilson, and Lincoln counties.
- The average unemployment rate across the 105 Kansas counties was approximately **3.4%**.

These results show that statewide or average measures can hide important differences between individual counties. Examining county-level data makes it possible to identify areas experiencing relatively greater labor-market challenges.

## Correlation Analysis

Correlation analysis is used to examine the direction and strength of relationships among the main economic and workforce indicators.

The analysis includes:

- Unemployment rate
- Poverty rate
- Median household income
- Bachelor's-or-higher educational attainment rate
- Labor-force participation rate

Correlation values range from **-1 to +1**. A positive value indicates that two variables tend to move in the same direction, while a negative value indicates that they tend to move in opposite directions. Correlation describes association and does not establish causation.

In [ ]:
# Select variables for correlation analysis
correlation_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

# Calculate the correlation matrix
correlation_matrix = df[correlation_columns].corr().round(2)

# Display the correlation matrix
correlation_matrix

In [ ]:
correlation_columns = [
    "unemployment_rate_bls",
    "poverty_rate",
    "median_household_income",
    "bachelors_or_higher_rate",
    "labor_force_participation_rate"
]

correlation_matrix = df[correlation_columns].corr().round(2)

correlation_matrix


## Multiple Linear Regression

A multiple linear regression model was used to examine how several county-level characteristics are associated with unemployment when considered together.

### Dependent Variable

- BLS unemployment rate

### Independent Variables

- Poverty rate
- Median household income
- Bachelor's-or-higher educational attainment rate
- Labor-force participation rate

The regression estimates the association between each predictor and unemployment while holding the other variables in the model constant.


In [ ]:
import statsmodels.api as sm

# Define the dependent variable
y = df["unemployment_rate_bls"]

# Define the independent variables
X = df[
    [
        "poverty_rate",
        "median_household_income",
        "bachelors_or_higher_rate",
        "labor_force_participation_rate"
    ]
]

# Add the regression intercept
X = sm.add_constant(X)

# Fit the multiple linear regression model
model = sm.OLS(y, X).fit()

# Display the regression results
model.summary()


### Regression Findings

The multiple linear regression model was statistically significant overall (**F-test p = 0.000660**) and explained approximately **17.5% of the variation in county unemployment rates (R² = 0.175)**.

Among the four predictors:

- **Labor-force participation rate** was statistically significant (**coefficient = -0.0378, p = 0.002**). Holding the other variables constant, a 1 percentage-point increase in labor-force participation was associated with approximately a 0.038 percentage-point decrease in the unemployment rate.
- **Poverty rate** had a positive coefficient (**0.0275**) but was not statistically significant at the 5% level (**p = 0.085**).
- **Median household income** was not statistically significant (**p = 0.590**).
- **Bachelor's-or-higher educational attainment** was not statistically significant (**p = 0.848**).

The results suggest that labor-force participation had the clearest independent association with county unemployment among the variables included in this model. However, the relatively low R² indicates that much of the variation in county unemployment is associated with factors not included in this model.

These results describe statistical associations and should not be interpreted as evidence of causal relationships.


## Multicollinearity Diagnostic

Variance Inflation Factor (VIF) was used to check whether the independent variables in the regression model were highly correlated with one another.

High multicollinearity can make regression coefficients unstable and difficult to interpret. As a general diagnostic guideline, VIF values below 5 indicate that serious multicollinearity is not evident.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each regression variable
vif_data = pd.DataFrame()

vif_data["Variable"] = X.columns

vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif_data

### VIF Findings

The VIF results indicate that multicollinearity is not a major concern in the regression model.

- Poverty rate: **VIF = 1.38**
- Median household income: **VIF = 1.78**
- Bachelor's-or-higher rate: **VIF = 1.32**
- Labor-force participation rate: **VIF = 1.33**

All predictor VIF values are well below 5. This suggests that the independent variables are not excessively correlated with one another and can reasonably be included together in the regression model.

The regression output also reported a large condition number. Because median household income is measured in dollars while the other predictors are percentages, differences in variable scale can contribute to a large condition number. The VIF results provide additional evidence that severe predictor multicollinearity is not apparent.


## Visual Analysis

The following visualizations provide additional context for the relationships identified in the descriptive, correlation, and regression analyses.

The charts examine:

- Poverty and median household income
- Unemployment and labor-force participation
- Educational attainment and household income
- Counties with the highest unemployment rates


In [ ]:
from IPython.display import Image, display

# Display poverty vs. median household income chart
display(
    Image(
        filename="../outputs/charts/poverty_vs_income.png"
    )
)

### Poverty and Household Income

The visualization shows a general negative relationship between poverty and median household income across Kansas counties.

This pattern is consistent with the correlation analysis, which found a correlation of **-0.49** between poverty rate and median household income. Counties with higher median household incomes generally tended to have lower poverty rates.

The relationship is not perfect, however, because counties with similar income levels can still have different poverty rates. This suggests that household income alone does not fully explain county-level differences in poverty.

The observed relationship represents an association and does not establish causation.

In [ ]:
# Display unemployment vs. labor-force participation chart
display(
    Image(
        filename="../outputs/charts/unemployment_vs_labor_force.png"
    )
)

In [ ]:
# Display educational attainment vs. household income chart
display(
    Image(
        filename="../outputs/charts/education_vs_income.png"
    )
)

### Educational Attainment and Household Income

The visualization shows a positive relationship between bachelor's-or-higher educational attainment and median household income across Kansas counties.

The correlation analysis found a correlation of **+0.40**, indicating that counties with higher educational attainment tended to have higher median household incomes.

For example, Johnson County had the highest bachelor's-or-higher rate among the counties identified in the ranking analysis at **57.56%**, along with a median household income of **$109,208**.

However, educational attainment was not statistically significant as an independent predictor of unemployment in the multiple regression model (**p = 0.848**). This demonstrates an important distinction: education may be associated with household income while not showing a statistically significant independent association with unemployment in this particular regression specification.

These findings represent county-level associations and do not establish causal relationships.

In [ ]:
# Display the Top 10 Kansas counties by unemployment rate
display(
    Image(
        filename="../outputs/charts/top_10_unemployment.png"
    )
)

### Counties with the Highest Unemployment Rates

The county ranking shows that unemployment conditions varied considerably across Kansas in the 2024 BLS LAUS data.

**Bourbon County** had the highest unemployment rate at **5.3%**, followed by **Woodson County** and **Linn County**, both at **4.9%**.

The average unemployment rate across all 105 counties was approximately **3.4%**, meaning the counties at the top of the ranking were notably above the county average.

Several counties with relatively high unemployment also appeared in the earlier SQL analysis of counties with both above-average unemployment and above-average poverty. This demonstrates the value of examining multiple economic indicators rather than relying on unemployment alone.

These county rankings are descriptive and identify areas that may warrant further investigation; they do not by themselves explain why unemployment differs across counties.

## SQL Analysis

The cleaned county-level dataset was loaded into a SQLite database to demonstrate SQL-based analysis.

The SQL analysis focused on four business and policy-oriented questions:

1. Which Kansas counties had the highest unemployment rates?
2. Which counties had both above-average unemployment and above-average poverty?
3. Which counties had the lowest labor-force participation rates?
4. Which counties had the highest bachelor's-or-higher educational attainment rates, and what were their median household incomes?

SQL complements the Python analysis by demonstrating how structured queries can be used to filter, rank, and compare county-level economic indicators.

In [ ]:
import sqlite3

# Connect to the project SQLite database
connection = sqlite3.connect(
    "../data/processed/kansas_economic_workforce.db"
)

# SQL query: Top 10 counties by unemployment rate
query = """
SELECT
    county_name,
    unemployment_rate
FROM kansas_county_economic_data
ORDER BY unemployment_rate DESC, county_name ASC
LIMIT 10;
"""

# Run the SQL query and display the results
sql_top_unemployment = pd.read_sql_query(query, connection)

sql_top_unemployment

In [ ]:
# SQL query: Counties with above-average unemployment
# and above-average poverty

query = """
SELECT
    county_name,
    unemployment_rate,
    poverty_rate
FROM kansas_county_economic_data
WHERE unemployment_rate > (
    SELECT AVG(unemployment_rate)
    FROM kansas_county_economic_data
)
AND poverty_rate > (
    SELECT AVG(poverty_rate)
    FROM kansas_county_economic_data
)
ORDER BY unemployment_rate DESC, poverty_rate DESC;
"""

sql_high_unemployment_poverty = pd.read_sql_query(
    query,
    connection
)

sql_high_unemployment_poverty

### SQL Findings: Unemployment and Poverty

The SQL analysis identified Kansas counties with both above-average unemployment and above-average poverty.

Examples included:

- Bourbon County
- Woodson County
- Geary County
- Cowley County
- Wyandotte County
- Wilson County
- Chautauqua County
- Brown County

This combined screening is more informative than ranking unemployment alone because it identifies counties experiencing challenges across multiple economic indicators.

The SQL approach also demonstrates how subqueries can be used to compare each county with statewide county-level averages and isolate areas that may warrant additional investigation.

In [ ]:
# SQL query: Counties with the lowest labor-force participation rates

query = """
SELECT
    county_name,
    labor_force_participation_rate,
    unemployment_rate,
    poverty_rate
FROM kansas_county_economic_data
ORDER BY labor_force_participation_rate ASC, county_name ASC
LIMIT 10;
"""

sql_lowest_participation = pd.read_sql_query(
    query,
    connection
)

sql_lowest_participation

### SQL Findings: Labor-Force Participation

The SQL results show that some Kansas counties had relatively low labor-force participation even when their unemployment rates were not especially high.

For example:

- **Norton County** had the lowest labor-force participation rate in the analysis at **45.97%**, while its unemployment rate was **2.9%**.
- **Ellsworth County** had labor-force participation of **50.76%** and unemployment of **3.8%**.
- **Woodson County** combined relatively low labor-force participation (**54.22%**) with relatively high unemployment (**4.9%**).

This illustrates why unemployment should not be interpreted by itself. A county can have a low unemployment rate while also having a relatively small share of its population participating in the labor force.

Examining both measures provides a more complete view of local workforce conditions.

In [ ]:
# SQL query: Counties with the highest educational attainment

query = """
SELECT
    county_name,
    bachelors_or_higher_rate,
    median_household_income
FROM kansas_county_economic_data
ORDER BY bachelors_or_higher_rate DESC,
         median_household_income DESC
LIMIT 10;
"""

sql_education_income = pd.read_sql_query(
    query,
    connection
)

sql_education_income

### SQL Findings: Educational Attainment and Income

The SQL ranking shows that counties with the highest educational attainment rates varied considerably in household income.

- **Johnson County** had the highest bachelor's-or-higher attainment rate at **57.56%** and a median household income of **$109,208**.
- **Douglas County** had an attainment rate of **53.19%** and median household income of **$69,746**.
- **Riley County** had an attainment rate of **48.91%** and median household income of **$61,098**.
- **Pottawatomie County** had an attainment rate of **38.19%** and median household income of **$92,325**.

The ranking is based primarily on educational attainment rather than household income. Therefore, these results should not be interpreted as a ranking of the highest-income Kansas counties.

Combined with the correlation analysis, the results show a positive county-level association between educational attainment and median household income, while also showing that counties with similar educational attainment levels can have different income levels.

## Key Findings

The analysis identified several important patterns in economic and workforce conditions across Kansas counties.

1. **Unemployment varied considerably across counties.**  
   The average county unemployment rate was approximately **3.4%**, while Bourbon County had the highest rate at **5.3%**. Woodson and Linn counties followed at **4.9%**.

2. **Labor-force participation was negatively associated with unemployment.**  
   The correlation between labor-force participation and unemployment was **-0.35**. In the multiple regression model, labor-force participation was the only predictor statistically significant at the 5% level (**coefficient = -0.0378, p = 0.002**).

3. **Low unemployment did not always indicate strong labor-force participation.**  
   Norton County had an unemployment rate of only **2.9%**, but its labor-force participation rate was **45.97%**, the lowest among the counties analyzed.

4. **Higher household income was associated with lower poverty.**  
   Poverty and median household income had a correlation of **-0.49**, one of the stronger relationships observed in the analysis.

5. **Educational attainment was positively associated with household income.**  
   Bachelor's-or-higher attainment and median household income had a correlation of **+0.40**. Johnson County had the highest educational attainment rate in the ranking at **57.56%** and a median household income of **$109,208**.

6. **The regression model explained only part of the differences in unemployment.**  
   The model had an **R² of 0.175**, indicating that the four included predictors explained approximately **17.5%** of the variation in county unemployment rates. Other economic, demographic, geographic, and industry-related factors may account for additional variation.

Overall, the findings demonstrate the importance of examining multiple indicators when evaluating local workforce conditions rather than relying on unemployment alone.

## Limitations

Several limitations should be considered when interpreting this analysis.

- **County-level analysis:** The project analyzes aggregated county data. Relationships observed at the county level should not be assumed to describe individual residents.

- **Cross-sectional analysis:** The analysis focuses primarily on 2024 conditions. It identifies differences and associations across counties but does not measure how these relationships change over time.

- **Correlation does not establish causation:** Relationships among unemployment, poverty, income, education, and labor-force participation represent statistical associations and should not be interpreted as causal effects.

- **Limited regression predictors:** The regression model includes four predictors and explains approximately **17.5%** of the variation in county unemployment rates. Other factors such as industry composition, demographics, commuting patterns, major employers, and local economic conditions were not included.

- **Different source methodologies:** Census ACS and BLS LAUS measures are produced using different methodologies. Although FIPS codes provide reliable geographic matching, similarly named economic measures from the two sources are not necessarily constructed in exactly the same way.

- **ACS estimates:** ACS values are survey-based estimates and are subject to sampling and measurement uncertainty.

These limitations mean the results are best used as an exploratory assessment of Kansas county economic and workforce conditions rather than as evidence of causal relationships.

## Conclusion

This project examined economic and workforce conditions across all 105 Kansas counties using 2024 Census ACS and BLS LAUS data.

The analysis showed meaningful differences in unemployment, poverty, household income, educational attainment, and labor-force participation across counties. Correlation analysis identified several relationships among these indicators, while multiple linear regression showed that labor-force participation had the clearest independent association with unemployment in the model.

The SQL analysis provided additional county-level screening by identifying areas with relatively high unemployment and poverty, low labor-force participation, and higher educational attainment.

Overall, the project demonstrates how **Python, SQL, statistical analysis, and data visualization** can be combined to investigate workforce and economic conditions. The results can be used as a starting point for identifying counties that may warrant further analysis, while recognizing that the findings are descriptive and do not establish causal relationships.